# 🧪 Context Provider Factory Test

### 🔁 Step 1: Reset and Seed Context Providers

In [ ]:
from pathlib import Path
import os, sys


# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: 22358acd-d5c9-4fa8-8fe3-3aa07fd3237d
Seeded SystemPrompt GUID: 33eaf901-74fe-435a-adf0-862f6e203acd
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.


### 🔍 Step 2: Retrieve Context Provider ID from DB

In [2]:
from sqlalchemy import select
from app.db.models import ContextProviderConfig

with Session(bind=engine) as session:
    context_record = session.execute(
        select(ContextProviderConfig).where(ContextProviderConfig.name == "Basic Context Provider")
    ).scalar_one()
    context_provider_id = context_record.id
    print("✅ Found Context Provider ID:", context_provider_id)

✅ Found Context Provider ID: 1


### 🏗️ Step 3: Instantiate Context Provider from Factory

In [3]:
from app.factories.context_provider_factory import ContextProviderFactory

provider = ContextProviderFactory.create(context_provider_id)
print("✅ ContextProvider instantiated:", provider.__class__.__name__)

✅ ContextProvider instantiated: BasicContextProvider


### 🧪 Step 4: Run get_context()

In [4]:
context = provider.get_context(experiment_id='test_exp_001', round=1)
print("✅ Retrieved Context:", context)

✅ Retrieved Context: {'file_path': 'tests/example.py', 'source_code': 'def example():\n    pass\n'}


### 📜 Step 5: Check Context Provider Logs

In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT * FROM context_provider_log WHERE experiment_id='test_exp_001'")
    ).fetchall()

assert rows, "❌ No context provider logs found."
print("✅ Logged Context Retrievals:")
for row in rows:
    print(row)

✅ Logged Context Retrievals:
(1, 'test_exp_001', 1, 'BasicContextProvider', '{"args": [], "kwargs": {}}', '{"file_path": "tests/example.py", "source_code": "def example():\\n    pass\\n"}', 1, None, '2025-05-24T17:59:32.403010+00:00')
